# 11 Bagging Tree Personalized Forecast
Eksperimen personalized step-by-step (satu model per notebook).


# Imports and Setup

Bagian ini memuat:
- Import library utama
- Konfigurasi untuk **PERSONALIZED forecasting** (1 model per user)
- Utility untuk evaluasi, tuning threshold, dan ringkasan per-user

- Semua nama kolom kini snake_case (mis. `user_id`, `stress_level`, `created_at`, `study_hour_per_day`).
- Kolom `is_restored` adalah metadata input/restore dan **tidak** dipakai sebagai fitur model.


In [ ]:
import os
os.environ["PYTHONWARNINGS"] = "ignore"

import warnings
warnings.simplefilter("ignore")

import json
import time
import numpy as np
import pandas as pd
from pathlib import Path
import joblib
import requests
import mlflow
import mlflow.sklearn
import importlib.util
from sqlalchemy import create_engine, text

from sklearn.metrics._classification import accuracy_score, f1_score
from sklearn.metrics._classification import accuracy_score, f1_score
from sklearn.model_selection import ParameterGrid

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    RandomForestClassifier, ExtraTreesClassifier,
    HistGradientBoostingClassifier,
    GradientBoostingClassifier, AdaBoostClassifier,
    BaggingClassifier
)
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV

from dotenv import load_dotenv
from pathlib import Path
load_dotenv(Path.cwd().joinpath("../../.env").resolve(), override=True)

# =============================================================================
# 0) CONFIG
# =============================================================================

# # Manual Train / Train Manual
# PARAMETERS = {
#     "data_source": "csv",
#     "dataset_path": "../../datasets/stress_forecast.csv",
#     "user_id": 0,
#     "window_size": 60,
#     "enable_eda": True,
#     # optional override:
#     # "output_path": "...",
#     # "secondary_output_path": "...",
# }
# DEFAULT_PARAMETERS = PARAMETERS
# PARAMETERS = globals().get("PARAMETERS") or DEFAULT_PARAMETERS

# Default Train
EXPERIMENT_NAME = "Personalized Forecast"
DEFAULT_PARAMETERS = {
    "data_source": "csv",
    "dataset_path": "../../datasets/stress_forecast.csv",
    "user_id": 1,
    "window_size": 60,
    "enable_eda": False,
    "output_path": "../../models/experiments_personalized/11_bagging_tree_personalized_forecast.joblib",
    "metrics_output_path": "../../models/experiments_personalized/11_bagging_tree_personalized_forecast_metrics.json",
}
PARAMETERS = {**DEFAULT_PARAMETERS, **(globals().get("PARAMETERS") or {})}
REGISTERED_MODEL_NAME = PARAMETERS.get("registered_model_name", "StressForecastPersonalized")
print("[experiment]", EXPERIMENT_NAME, "| user_id=", PARAMETERS.get("user_id"))


ENABLE_EDA = PARAMETERS.get("enable_eda", False)
data_source = PARAMETERS.get("data_source", os.getenv("DATA_SOURCE", "db"))
user_id = PARAMETERS.get("user_id")
window_size = int(PARAMETERS.get("window_size", 60) or 60)

# primary output (existing)
output_path = PARAMETERS.get("output_path")

# secondary output (dinamis: PARAMETERS > env)
secondary_output_path = PARAMETERS.get(
    "secondary_output_path",
    os.getenv("SECONDARY_OUTPUT_PATH")
)

metrics_output_path = PARAMETERS.get("metrics_output_path")
dataset_path = PARAMETERS.get("dataset_path")
streak_start_date = PARAMETERS.get("streak_start_date")

if window_size != 60:
    print("window_size override diabaikan. Sistem menggunakan fixed 60.")
    window_size = 60

def _resolve_repo_root() -> Path:
    cwd = Path.cwd().resolve()
    for parent in [cwd, *cwd.parents]:
        if (parent / "nostressia-machine-learning").exists() and (parent / "nostressia-backend").exists():
            return parent
    return cwd

repo_root = _resolve_repo_root()
UTILS_PATH = repo_root / "nostressia-machine-learning" / "Stress-Forecast" / "notebooks" / "mlflow_utils.py"
if not UTILS_PATH.exists():
    raise FileNotFoundError(f"mlflow_utils.py tidak ditemukan: {UTILS_PATH}")
spec = importlib.util.spec_from_file_location("sf_mlflow_utils", UTILS_PATH)
sf_utils = importlib.util.module_from_spec(spec)
spec.loader.exec_module(sf_utils)
configure_mlflow = sf_utils.configure_mlflow
ensure_notebook_run = sf_utils.ensure_notebook_run

repo_root, tracking_uri = configure_mlflow(EXPERIMENT_NAME)
mlflow.sklearn.autolog(log_models=False, silent=True)
RUN_NAME_DEFAULT = f"{Path(PARAMETERS.get('output_path', 'forecast_experiment')).stem.replace('_', ' ').title()}"
RUN_NAME = PARAMETERS.get("run_name", RUN_NAME_DEFAULT)
ACTIVE_RUN = ensure_notebook_run(RUN_NAME)
print(f"[mlflow] tracking_uri={tracking_uri} | experiment={EXPERIMENT_NAME} | run_id={ACTIVE_RUN.info.run_id}")

# primary default (existing)
default_model_out = repo_root / "nostressia-backend" / "app" / "models_ml" / "personalized_forecast.joblib"
MODEL_OUT = Path(output_path) if output_path else default_model_out

# secondary default (ngikutin cara: repo_root + relative path)
default_secondary_model_out = (
    repo_root
    / "nostressia-machine-learning"
    / "Stress-Forecast"
    / "models"
    / "personalized_forecast.joblib"
)

SECOND_MODEL_OUT = (
    Path(secondary_output_path)
    if secondary_output_path
    else default_secondary_model_out
)

DB_HOST = os.getenv("DB_HOST")
DB_PORT = os.getenv("DB_PORT", "3306")
DB_USER = os.getenv("DB_USER")
DB_PASSWORD = os.getenv("DB_PASSWORD")
DB_NAME = os.getenv("DB_NAME")

BACKEND_BASE_URL = os.getenv("BACKEND_BASE_URL")
INTERNAL_TOKEN = os.getenv("INTERNAL_TOKEN")

DATE_COL   = "date"
USER_COL   = "user_id"
TARGET_COL = "stress_level"  # 0..2

WINDOW   = 3
TEST_LEN = 12

VAL_WINDOWS = [(10, 20), (15, 25)]
THRESHOLDS  = np.linspace(0.05, 0.95, 19)

RANDOM_STATE = 26

# Personalized: default = False (no semi-global/global)
USE_USER_ID_FEATURE = False

# Threshold tuning:
# - True  => tune threshold per user (fair: uses only that user's CV folds)
# - False => tune one global threshold pooled across users
TUNE_THRESHOLD_PER_USER = True

# Print per-user details for baselines and selected best model
PRINT_PER_USER_DETAILS = True


# =============================================================================
# Print helpers
# =============================================================================
def kv(k, v):
    print(f"{k:<20}: {v}")

def safe_class_counts(y):
    y = np.asarray(y).astype(int)
    return {0: int((y == 0).sum()), 1: int((y == 1).sum())}

def print_per_user_breakdown(title, per_user_records, thr_info=None):
    print(f"\n{title}")
    for r in per_user_records:
        uid = r["uid"]
        y = np.asarray(r["y"]).astype(int)
        pred = np.asarray(r["pred"]).astype(int)
        acc = float(accuracy_score(y, pred))
        f1  = float(f1_score(y, pred, zero_division=0))
        dist = safe_class_counts(y)

        extra = ""
        if thr_info is not None:
            if isinstance(thr_info, dict) and uid in thr_info:
                extra = f" | thr={float(thr_info[uid]):.2f}"
            elif isinstance(thr_info, (float, int)):
                extra = f" | thr={float(thr_info):.2f}"

        print(f"uid={uid} | n={len(y):<3} | dist={dist} | acc={acc:.4f} | f1={f1:.4f}{extra}")


# =============================================================================
# Core helpers
# =============================================================================
def eval_bin(y_true, y_pred):
    return {
        "acc": float(accuracy_score(y_true, y_pred)),
        "f1":  float(f1_score(y_true, y_pred, zero_division=0)),
    }

def tune_thr_from_proba(y_true, p_high, thresholds=THRESHOLDS):
    best_thr, best_f1 = None, -1.0
    for thr in thresholds:
        pred = (p_high >= thr).astype(int)
        f1 = float(f1_score(y_true, pred, zero_division=0))
        if f1 > best_f1:
            best_f1, best_thr = f1, thr
    return float(best_thr), float(best_f1)

def per_user_macro_metrics(per_user_records):
    accs, f1s = [], []
    for r in per_user_records:
        accs.append(accuracy_score(r["y"], r["pred"]))
        f1s.append(f1_score(r["y"], r["pred"], zero_division=0))
    return float(np.mean(accs)), float(np.mean(f1s))

def cv_folds_user(tp_df):
    folds = []
    for (v0, v1) in VAL_WINDOWS:
        if len(tp_df) < v1:
            continue
        tr = tp_df.iloc[:v0].copy()
        va = tp_df.iloc[v0:v1].copy()
        folds.append((tr, va))
    return folds

def min_class_count(y):
    vc = pd.Series(np.asarray(y)).value_counts()
    if len(vc) < 2:
        return 0
    return int(vc.min())


# Load and Explore Dataset

Bagian ini mencakup:
- Load dataset
- Validasi kolom wajib
- Parsing `date` dan sorting time-series per user
- Validasi range target `stress_level` (0–2)


In [ ]:
# LOAD REALTIME DATA

# LOAD FROM CSV
def _load_personalized_from_csv(path, target_user_id, start_date):
    if not path:
        raise RuntimeError("dataset_path wajib untuk data_source=csv.")
    df = pd.read_csv(path)
    
    # Default Train
    df = df[df["user_id"] == int(target_user_id)]

    # # Manual Train / Train Manual
    # df = df.sort_values(["user_id", "date"]).groupby("user_id").head(60)

    if start_date:
        df = df[pd.to_datetime(df["date"]) >= pd.to_datetime(start_date)]
    return df

if user_id is None:
    raise ValueError("user_id parameter wajib untuk training personalized.")

def _load_personalized_from_db(target_user_id, limit):
    if not all([DB_HOST, DB_USER, DB_PASSWORD, DB_NAME]):
        raise RuntimeError("DB config tidak lengkap untuk load data personalized.")
    url = (
        f"mysql+mysqlconnector://{DB_USER}:{DB_PASSWORD}"
        f"@{DB_HOST}:{DB_PORT}/{DB_NAME}"
    )
    engine = create_engine(url)
    query = """
        SELECT
            user_id,
            date,
            stress_level,
            gpa,
            extracurricular_hour_per_day,
            physical_activity_hour_per_day,
            sleep_hour_per_day,
            study_hour_per_day,
            social_hour_per_day,
            emoji,
            is_restored
        FROM stress_levels
        WHERE user_id = :user_id
        ORDER BY date DESC
        LIMIT :limit
    """
    with engine.connect() as conn:
        return pd.read_sql(text(query), conn, params={"user_id": int(target_user_id), "limit": int(limit)})

def _load_personalized_from_api(target_user_id, limit):
    if not BACKEND_BASE_URL:
        raise RuntimeError("BACKEND_BASE_URL belum di-set untuk mode API.")
    url = f"{BACKEND_BASE_URL.rstrip('/')}/api/ml/training-data/personalized"
    headers = {}
    if INTERNAL_TOKEN:
        headers["X-Internal-Token"] = INTERNAL_TOKEN
    params = {"userId": int(target_user_id), "limit": int(limit)}
    resp = requests.get(url, headers=headers, params=params, timeout=30)
    resp.raise_for_status()
    payload = resp.json()
    return pd.DataFrame(payload.get("data", []))

if data_source == "db":
    df = _load_personalized_from_db(user_id, window_size)
elif data_source == "api":
    df = _load_personalized_from_api(user_id, window_size)
elif data_source == "csv":
    df = _load_personalized_from_csv(dataset_path, user_id, streak_start_date)
else:
    raise ValueError(f"Unknown data_source: {data_source}")

if df.empty:
    raise ValueError("SKIPPED_INSUFFICIENT_DATA: personalized dataset kosong.")

if "is_restored" not in df.columns:
    df["is_restored"] = 0
df["is_restored"] = df["is_restored"].fillna(0).astype(int)

for required_col in [DATE_COL, USER_COL, TARGET_COL]:
    if required_col not in df.columns:
        raise KeyError(f"Required column '{required_col}' not found in dataset.")

df[DATE_COL] = pd.to_datetime(df[DATE_COL], errors="raise")
df = df.sort_values([USER_COL, DATE_COL]).reset_index(drop=True)

if not df[TARGET_COL].dropna().between(0, 2).all():
    raise ValueError(f"'{TARGET_COL}' must be within range 0..2")

if len(df) < 14:
    raise ValueError("SKIPPED_INSUFFICIENT_DATA: data personalized kurang dari 14 baris.")

kv("DATA_SOURCE", data_source)
kv("ROWS_RAW", len(df))
kv("USERS_RAW", df[USER_COL].nunique())
kv("DATE_RANGE_RAW", f"{df[DATE_COL].min().date()} -> {df[DATE_COL].max().date()}")

print("\nHEAD:")
display(df.head())


# Exploratory Data Analysis (EDA)

Ringkasan cepat:
- Distribusi `stress_level` (0–2)
- Jumlah baris per user (indikasi kecukupan data untuk split & CV)


In [ ]:
# EDA QUICK CHECKS
if ENABLE_EDA:
    import matplotlib.pyplot as plt

    target_counts = df[TARGET_COL].value_counts().sort_index()
    print("TARGET_DIST (0..2):", target_counts.to_dict())

    plt.figure()
    target_counts.plot(kind="bar")
    plt.title(f"Distribution of {TARGET_COL} (0..2)")
    plt.xlabel(TARGET_COL)
    plt.ylabel("count")
    plt.show()

    user_counts = df[USER_COL].value_counts()
    print("\nROWS_PER_USER:", user_counts.to_dict())

    plt.figure()
    user_counts.sort_index().plot(kind="bar")
    plt.title("Rows per user")
    plt.xlabel(USER_COL)
    plt.ylabel("count")
    plt.show()


# Data Preprocessing

Bagian ini melakukan **feature engineering tanpa leakage**:
- Fitur kalender: `dow`, `is_weekend`
- Fitur lag target: `lag_sp_1..lag_sp_WINDOW`
- Rolling stats dari history yang berakhir di `t-1`

Transformasi label:
- `y_bin = 1` jika `stress_level >= 1`
- `y_bin = 0` jika `stress_level == 0`


In [ ]:
# FEATURE ENGINEERING (NO-LEAK)
rows = []
for uid, g in df.groupby(USER_COL):
    g = g.sort_values(DATE_COL).reset_index(drop=True)

    g["dow"] = g[DATE_COL].dt.dayofweek.astype(int)
    g["is_weekend"] = (g["dow"] >= 5).astype(int)

    for k in range(1, WINDOW + 1):
        g[f"lag_sp_{k}"] = g[TARGET_COL].shift(k)

    sp_shift = g[TARGET_COL].shift(1)

    g["sp_mean"] = sp_shift.rolling(WINDOW).mean()
    g["sp_std"]  = sp_shift.rolling(WINDOW).std().fillna(0.0)
    g["sp_min"]  = sp_shift.rolling(WINDOW).min()
    g["sp_max"]  = sp_shift.rolling(WINDOW).max()

    g["count_high"] = (sp_shift >= 1).rolling(WINDOW).sum()
    g["count_low"]  = (sp_shift == 0).rolling(WINDOW).sum()

    high = (sp_shift >= 1).astype(int).fillna(0).astype(int).tolist()
    streak, cur = [], 0
    for v in high:
        cur = cur + 1 if v == 1 else 0
        streak.append(cur)
    g["streak_high"] = streak

    diff = (sp_shift != sp_shift.shift(1)).astype(int)
    g["transitions"] = diff.rolling(WINDOW).sum()

    rows.append(g)

feat = pd.concat(rows, ignore_index=True)

# Binary labeling: y_bin = 1 if pred>=1 else 0
feat["y_bin"] = (feat[TARGET_COL] >= 1).astype(int)

feature_cols = (
    ["dow", "is_weekend"]
    + [f"lag_sp_{k}" for k in range(1, WINDOW + 1)]
    + [
        "sp_mean", "sp_std", "sp_min", "sp_max",
        "count_high", "count_low",
        "streak_high", "transitions",
    ]
)
if USE_USER_ID_FEATURE:
    feature_cols = [USER_COL] + feature_cols

feat = feat.dropna(subset=feature_cols + ["y_bin"]).reset_index(drop=True)
users = sorted(feat[USER_COL].unique().tolist())

kv("ROWS_FEAT", len(feat))
kv("USERS", users)
kv("DATE_RANGE_FEAT", f"{feat[DATE_COL].min().date()} -> {feat[DATE_COL].max().date()}")
kv("WINDOW", WINDOW)
kv("TEST_LEN", TEST_LEN)
kv("FEATURES_COUNT", len(feature_cols))
kv("USE_USER_ID_FEATURE", USE_USER_ID_FEATURE)
kv("BINARY_DIST", feat["y_bin"].value_counts().to_dict())
kv("VAL_WINDOWS", VAL_WINDOWS)
kv("TUNE_THRESHOLD_PER_USER", TUNE_THRESHOLD_PER_USER)

display(feat[[USER_COL, DATE_COL, TARGET_COL, "y_bin"] + feature_cols].head())


# Model Training and Evaluation

Cakupan bagian ini:
- Split time-based per user (TEST = last `TEST_LEN`)
- Baseline L1: Persistence
- Baseline L2: Markov per-user + threshold tuning (pooled CV)
- Kandidat model ML per user (tanpa user_id sebagai fitur)
- SVM calibrated dengan cv adaptif (SAFE)
- Leaderboard + seleksi model terbaik vs baseline
- Simpan artifact ke file `.joblib`


In [ ]:
# SPLIT PER USER (TIME-BASED)
per_user = {}
split_rows = []

for uid in users:
    g = feat[feat[USER_COL] == uid].sort_values(DATE_COL).reset_index(drop=True)
    n = len(g)
    test_start = n - TEST_LEN
    if test_start <= 10:
        raise ValueError(f"User {uid}: insufficient rows for split (n={n}, TEST_LEN={TEST_LEN}).")

    tp = g.iloc[:test_start].copy()
    te = g.iloc[test_start:].copy()

    per_user[uid] = {"train_pool": tp, "test": te}

    split_rows.append({
        "uid": uid,
        "n_total": n,
        "n_train_pool": len(tp),
        "n_test": len(te),
        "train_pool_dist": safe_class_counts(tp["y_bin"].values),
        "test_dist": safe_class_counts(te["y_bin"].values),
    })

kv("TOTAL_TRAINPOOL", sum(r["n_train_pool"] for r in split_rows))
kv("TOTAL_TEST", sum(r["n_test"] for r in split_rows))

print("\nPER_USER_SPLIT:")
for r in split_rows:
    print(
        f"uid={r['uid']} | total={r['n_total']} | train_pool={r['n_train_pool']} dist={r['train_pool_dist']} "
        f"| test={r['n_test']} dist={r['test_dist']}"
    )


In [ ]:
# BASELINE L1: PERSISTENCE (PER USER)
persist_user_records = []
all_true, all_pred = [], []

for uid in users:
    te = per_user[uid]["test"]
    y = te["y_bin"].astype(int).values
    pred = (te["lag_sp_1"] >= 1).astype(int).values

    persist_user_records.append({"uid": uid, "y": y, "pred": pred})
    all_true.append(y)
    all_pred.append(pred)

y_all = np.concatenate(all_true)
pred_all = np.concatenate(all_pred)

persist_pooled = eval_bin(y_all, pred_all)
persist_macro_acc, persist_macro_f1 = per_user_macro_metrics(persist_user_records)

kv("TEST_POOLED_ACC", persist_pooled["acc"])
kv("TEST_POOLED_F1", persist_pooled["f1"])
kv("TEST_MACRO_ACC", persist_macro_acc)
kv("TEST_MACRO_F1", persist_macro_f1)

if PRINT_PER_USER_DETAILS:
    print_per_user_breakdown("PER-USER (Persistence) on TEST:", persist_user_records)


In [ ]:
# BASELINE L2: MARKOV PER USER (prev_high, dow) + THRESHOLD TUNING
def train_markov_one_user(df_train):
    counts = np.zeros((2, 7, 2), dtype=int)  # prev(2) x dow(7) x y(2)
    prev = (df_train["lag_sp_1"] >= 1).astype(int).values
    dow  = (df_train["dow"]).astype(int).values
    yb   = (df_train["y_bin"]).astype(int).values
    for p, d, y in zip(prev, dow, yb):
        counts[p, d, y] += 1
    probs = (counts + 1) / (counts.sum(axis=2, keepdims=True) + 2)  # Laplace smoothing
    return probs

def markov_proba_user(probs, df_eval):
    prev = (df_eval["lag_sp_1"] >= 1).astype(int).values
    dow  = (df_eval["dow"]).astype(int).values
    return np.array([probs[p, d, 1] for p, d in zip(prev, dow)], dtype=float)

# Pooled CV threshold tuning (fair: uses only train_pool folds)
cv_true, cv_phigh = [], []
cv_fold_stats = []

for uid in users:
    tp = per_user[uid]["train_pool"]
    folds = cv_folds_user(tp)
    for (tr_df, va_df) in folds:
        probs = train_markov_one_user(tr_df)
        p = markov_proba_user(probs, va_df)
        cv_true.append(va_df["y_bin"].astype(int).values)
        cv_phigh.append(p)
        cv_fold_stats.append({"uid": uid, "tr_len": len(tr_df), "va_len": len(va_df)})

if len(cv_true) == 0:
    raise ValueError("No valid CV folds. Reduce VAL_WINDOWS / TEST_LEN / WINDOW.")

cv_true = np.concatenate(cv_true)
cv_phigh = np.concatenate(cv_phigh)

thr_mk, cv_f1_mk = tune_thr_from_proba(cv_true, cv_phigh)

mk_models = {}
markov_user_records = []
all_true, all_pred = [], []

for uid in users:
    tp = per_user[uid]["train_pool"]
    te = per_user[uid]["test"]

    probs = train_markov_one_user(tp)
    mk_models[uid] = probs

    p = markov_proba_user(probs, te)
    pred = (p >= thr_mk).astype(int)
    y = te["y_bin"].astype(int).values

    markov_user_records.append({"uid": uid, "y": y, "pred": pred})
    all_true.append(y)
    all_pred.append(pred)

y_all = np.concatenate(all_true)
pred_all = np.concatenate(all_pred)

markov_pooled = eval_bin(y_all, pred_all)
markov_macro_acc, markov_macro_f1 = per_user_macro_metrics(markov_user_records)

kv("CV_FOLDS_TOTAL", len(cv_fold_stats))
kv("CV_POOLED_DIST", safe_class_counts(cv_true))
kv("BEST_THR_MARKOV", thr_mk)
kv("CV_POOLED_F1", cv_f1_mk)
kv("TEST_POOLED_ACC", markov_pooled["acc"])
kv("TEST_POOLED_F1", markov_pooled["f1"])
kv("TEST_MACRO_ACC", markov_macro_acc)
kv("TEST_MACRO_F1", markov_macro_f1)

if PRINT_PER_USER_DETAILS:
    print_per_user_breakdown("PER-USER (Markov) on TEST:", markov_user_records, thr_info=thr_mk)


In [ ]:
# PREPROCESS (FOR ML)
cat_cols = ["dow", "is_weekend"]
if USE_USER_ID_FEATURE:
    cat_cols = [USER_COL] + cat_cols

num_cols = [c for c in feature_cols if c not in cat_cols]

preprocess = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
        ("num", Pipeline([("imp", SimpleImputer(strategy="median"))]), num_cols),
    ],
    remainder="drop",
)

kv("CAT_COLS", cat_cols)
kv("NUM_COLS_COUNT", len(num_cols))


In [ ]:
# CANDIDATE MODELS
try:
    bag_base = BaggingClassifier(estimator=DecisionTreeClassifier(random_state=RANDOM_STATE), random_state=RANDOM_STATE, n_jobs=1)
    BAG_ESTIMATOR_PARAM = "clf__estimator__"
except TypeError:
    bag_base = BaggingClassifier(base_estimator=DecisionTreeClassifier(random_state=RANDOM_STATE), random_state=RANDOM_STATE, n_jobs=1)
    BAG_ESTIMATOR_PARAM = "clf__base_estimator__"
CANDIDATES = {
    "BaggingTree": (
        bag_base,
        {"clf__n_estimators": [20, 50, 100], f"{BAG_ESTIMATOR_PARAM}max_depth": [None, 6, 10], f"{BAG_ESTIMATOR_PARAM}min_samples_leaf": [1, 2, 4]},
    ),
}
print("[experiment] Single-model mode: BaggingTree")


In [ ]:
# TUNING UTILITIES (GLOBAL THRESHOLD OR PER-USER THRESHOLD)
def tune_global_thr_pooled_over_all_users(pipe, params):
    y_list, p_list = [], []
    for uid in users:
        tp = per_user[uid]["train_pool"]
        folds = cv_folds_user(tp)
        for tr_df, va_df in folds:
            ytr = tr_df["y_bin"].astype(int).values
            if len(np.unique(ytr)) < 2:
                continue
            pipe.set_params(**params)
            pipe.fit(tr_df[feature_cols], ytr)
            p = pipe.predict_proba(va_df[feature_cols])[:, 1]
            y_list.append(va_df["y_bin"].astype(int).values)
            p_list.append(p)

    if len(y_list) == 0:
        return None, None

    y_all = np.concatenate(y_list)
    p_all = np.concatenate(p_list)
    thr, cv_f1 = tune_thr_from_proba(y_all, p_all)
    return float(thr), float(cv_f1)

def tune_per_user_thr(pipe, params):
    thr_by_user = {}
    f1s = []

    for uid in users:
        tp = per_user[uid]["train_pool"]
        folds = cv_folds_user(tp)
        if len(folds) == 0:
            return None, None

        y_list, p_list = [], []
        for tr_df, va_df in folds:
            ytr = tr_df["y_bin"].astype(int).values
            if len(np.unique(ytr)) < 2:
                continue
            pipe.set_params(**params)
            pipe.fit(tr_df[feature_cols], ytr)
            p = pipe.predict_proba(va_df[feature_cols])[:, 1]
            y_list.append(va_df["y_bin"].astype(int).values)
            p_list.append(p)

        if len(y_list) == 0:
            return None, None

        y_u = np.concatenate(y_list)
        p_u = np.concatenate(p_list)
        thr_u, f1_u = tune_thr_from_proba(y_u, p_u)
        thr_by_user[uid] = float(thr_u)
        f1s.append(float(f1_u))

    return thr_by_user, float(np.mean(f1s))

def eval_personalized_models(models_by_user, thr_by_user_or_scalar):
    per_user_records = []
    all_true, all_pred = [], []

    for uid in users:
        te = per_user[uid]["test"]
        y = te["y_bin"].astype(int).values

        pipe = models_by_user[uid]
        p = pipe.predict_proba(te[feature_cols])[:, 1]
        thr = thr_by_user_or_scalar[uid] if isinstance(thr_by_user_or_scalar, dict) else float(thr_by_user_or_scalar)
        pred = (p >= thr).astype(int)

        per_user_records.append({"uid": uid, "y": y, "pred": pred})
        all_true.append(y)
        all_pred.append(pred)

    y_all = np.concatenate(all_true)
    pred_all = np.concatenate(all_pred)

    pooled = eval_bin(y_all, pred_all)
    macro_acc, macro_f1 = per_user_macro_metrics(per_user_records)
    macro = {"acc": float(macro_acc), "f1": float(macro_f1)}
    return pooled, macro, per_user_records


In [ ]:
# PERSONALIZED ML: TRAIN + TUNE (NON-SVM)
rows = []

for name, (clf, grid) in CANDIDATES.items():
    best = None

    for params in ParameterGrid(grid):
        pipe = Pipeline([("prep", preprocess), ("clf", clf)])

        if TUNE_THRESHOLD_PER_USER:
            thr_obj, cv_score = tune_per_user_thr(pipe, params)
        else:
            thr_obj, cv_score = tune_global_thr_pooled_over_all_users(pipe, params)

        if thr_obj is None:
            continue

        if (best is None) or (cv_score > best["cv_score"]):
            best = {"params": dict(params), "thr_obj": thr_obj, "cv_score": float(cv_score)}

    if best is None:
        print(f"SKIP_MODEL: {name} (no valid params/folds)")
        continue

    # Final training per user (train_pool only)
    models_by_user = {}
    ok = True

    for uid in users:
        tp = per_user[uid]["train_pool"]
        ytr = tp["y_bin"].astype(int).values
        if len(np.unique(ytr)) < 2:
            ok = False
            break

        pipe = Pipeline([("prep", preprocess), ("clf", clf)])
        pipe.set_params(**best["params"])
        pipe.fit(tp[feature_cols], ytr)
        models_by_user[uid] = pipe

    if not ok:
        print(f"SKIP_MODEL: {name} (some user train_pool has single class)")
        continue

    pooled, macro, user_records = eval_personalized_models(models_by_user, best["thr_obj"])

    rows.append({
        "model": name,
        "cv_score": float(best["cv_score"]),
        "thr_obj": best["thr_obj"],
        "test_pooled_f1": float(pooled["f1"]),
        "test_pooled_acc": float(pooled["acc"]),
        "test_macro_f1": float(macro["f1"]),
        "test_macro_acc": float(macro["acc"]),
        "params": dict(best["params"]),
        "models_by_user": models_by_user,
        "test_user_records": user_records,
    })

    thr_desc = "per-user" if isinstance(best["thr_obj"], dict) else f"{best['thr_obj']:.2f}"
    print(f"\nMODEL: {name}")
    kv("CV_SCORE", best["cv_score"])
    kv("THRESHOLD", thr_desc)
    kv("TEST_POOLED_F1", pooled["f1"])
    kv("TEST_POOLED_ACC", pooled["acc"])
    kv("TEST_MACRO_F1", macro["f1"])
    kv("TEST_MACRO_ACC", macro["acc"])
    kv("PARAMS", best["params"])


In [ ]:
SVM_NAME = "LinearSVC(calibrated)"
SVM_GRID = PARAMETERS.get("svm_grid", {"C": [0.03, 0.1, 0.3, 1.0, 3.0]})
# SVM CALIBRATED SAFE (ADAPTIVE CV PER FOLD)
def make_calibrator(base, cv_k):
    try:
        return CalibratedClassifierCV(estimator=base, method="sigmoid", cv=cv_k)
    except TypeError:
        return CalibratedClassifierCV(base_estimator=base, method="sigmoid", cv=cv_k)

def svm_fit_predict_proba(tr_X, tr_y, va_X, C, cv_max=3):
    mcc = min_class_count(tr_y)
    cv_k = int(min(cv_max, mcc))
    if cv_k < 2:
        return None, cv_k
    base = LinearSVC(class_weight="balanced", random_state=RANDOM_STATE, C=float(C))
    calib = make_calibrator(base, cv_k=cv_k)
    pipe = Pipeline([("prep", preprocess), ("clf", calib)])
    pipe.fit(tr_X, tr_y)
    return pipe.predict_proba(va_X)[:, 1], cv_k

# Feasibility check for final training across all users
svm_feasible_all_users = True
svm_feasible_detail = []

for uid in users:
    y_tp = per_user[uid]["train_pool"]["y_bin"].astype(int).values
    mcc = min_class_count(y_tp)
    svm_feasible_detail.append({"uid": uid, "min_class_count_trainpool": mcc})
    if mcc < 2:
        svm_feasible_all_users = False

kv("SVM_FEASIBLE_ALL_USERS", svm_feasible_all_users)
print("SVM_FEASIBLE_DETAIL:")
for r in svm_feasible_detail:
    print(f"uid={r['uid']} | min_class_count_trainpool={r['min_class_count_trainpool']}")

if svm_feasible_all_users:
    best = None

    for C in SVM_GRID["C"]:
        if TUNE_THRESHOLD_PER_USER:
            thr_by_user = {}
            per_user_cv_scores = []
            all_users_ok = True
            users_valid = 0

            for uid in users:
                tp = per_user[uid]["train_pool"]
                folds = cv_folds_user(tp)

                y_list_u, p_list_u = [], []
                for (tr_df, va_df) in folds:
                    tr_y = tr_df["y_bin"].astype(int).values
                    p, cv_k = svm_fit_predict_proba(tr_df[feature_cols], tr_y, va_df[feature_cols], C=C, cv_max=3)
                    if p is None:
                        continue
                    y_list_u.append(va_df["y_bin"].astype(int).values)
                    p_list_u.append(p)

                if len(y_list_u) == 0:
                    all_users_ok = False
                    break

                y_u = np.concatenate(y_list_u)
                p_u = np.concatenate(p_list_u)
                thr_u, f1_u = tune_thr_from_proba(y_u, p_u)
                thr_by_user[uid] = float(thr_u)
                per_user_cv_scores.append(float(f1_u))
                users_valid += 1

            if (not all_users_ok) or (users_valid < len(users)):
                continue

            cv_score = float(np.mean(per_user_cv_scores))
            thr_obj = thr_by_user

        else:
            y_list, p_list = [], []
            for uid in users:
                tp = per_user[uid]["train_pool"]
                folds = cv_folds_user(tp)
                for (tr_df, va_df) in folds:
                    tr_y = tr_df["y_bin"].astype(int).values
                    p, cv_k = svm_fit_predict_proba(tr_df[feature_cols], tr_y, va_df[feature_cols], C=C, cv_max=3)
                    if p is None:
                        continue
                    y_list.append(va_df["y_bin"].astype(int).values)
                    p_list.append(p)

            if len(y_list) == 0:
                continue

            y_all = np.concatenate(y_list)
            p_all = np.concatenate(p_list)
            thr_obj, cv_score = tune_thr_from_proba(y_all, p_all)

        if (best is None) or (cv_score > best["cv_score"]):
            best = {"C": float(C), "thr_obj": thr_obj, "cv_score": float(cv_score)}

    if best is None:
        print(f"SKIP_MODEL: {SVM_NAME} (no valid C across all users/folds)")
    else:
        # Final training per user (adaptive cv from train_pool)
        models_by_user = {}
        ok = True
        final_cv_by_user = {}

        for uid in users:
            tp = per_user[uid]["train_pool"]
            tr_y = tp["y_bin"].astype(int).values
            mcc = min_class_count(tr_y)
            cv_k = int(min(3, mcc))
            if cv_k < 2:
                ok = False
                break

            base = LinearSVC(class_weight="balanced", random_state=RANDOM_STATE, C=float(best["C"]))
            calib = make_calibrator(base, cv_k=cv_k)
            pipe = Pipeline([("prep", preprocess), ("clf", calib)])
            pipe.fit(tp[feature_cols], tr_y)

            models_by_user[uid] = pipe
            final_cv_by_user[uid] = cv_k

        if not ok:
            print(f"SKIP_MODEL: {SVM_NAME} (final training not feasible for all users)")
        else:
            pooled, macro, user_records = eval_personalized_models(models_by_user, best["thr_obj"])
            rows.append({
                "model": SVM_NAME,
                "cv_score": float(best["cv_score"]),
                "thr_obj": best["thr_obj"],
                "test_pooled_f1": float(pooled["f1"]),
                "test_pooled_acc": float(pooled["acc"]),
                "test_macro_f1": float(macro["f1"]),
                "test_macro_acc": float(macro["acc"]),
                "params": {"C": float(best["C"]), "calibration_cv": f"adaptive<=3 (per user), {final_cv_by_user}"},
                "models_by_user": models_by_user,
                "test_user_records": user_records,
            })

            thr_desc = "per-user" if isinstance(best["thr_obj"], dict) else f"{best['thr_obj']:.2f}"
            print(f"\nMODEL: {SVM_NAME}")
            kv("CV_SCORE", best["cv_score"])
            kv("THRESHOLD", thr_desc)
            kv("TEST_POOLED_F1", pooled["f1"])
            kv("TEST_POOLED_ACC", pooled["acc"])
            kv("TEST_MACRO_F1", macro["f1"])
            kv("TEST_MACRO_ACC", macro["acc"])
            kv("PARAMS", {"C": best["C"], "final_cv_by_user": final_cv_by_user})
else:
    print(f"SKIP_MODEL: {SVM_NAME} (some user train_pool has single class)")


In [ ]:
# LEADERBOARD + SELECT BEST (VS MARKOV)
print("BASELINES:")
print(
    f"  Baseline-Persist | TEST pooled: acc={persist_pooled['acc']:.4f}, f1={persist_pooled['f1']:.4f} | "
    f"macro(user): acc={persist_macro_acc:.4f}, f1={persist_macro_f1:.4f}"
)
print(
    f"  Baseline-Markov  | CV pooled: f1={cv_f1_mk:.4f}, thr={thr_mk:.2f} | "
    f"TEST pooled: acc={markov_pooled['acc']:.4f}, f1={markov_pooled['f1']:.4f} | "
    f"macro(user): acc={markov_macro_acc:.4f}, f1={markov_macro_f1:.4f}"
)

rows_sorted = sorted(rows, key=lambda r: r["test_pooled_f1"], reverse=True)

print("\nCANDIDATES:")
if len(rows_sorted) == 0:
    print("  (no ML candidates succeeded)")
else:
    for r in rows_sorted:
        thr_desc = "per-user" if isinstance(r["thr_obj"], dict) else f"{r['thr_obj']:.2f}"
        print(
            f"  {r['model']:<26} | CV={r['cv_score']:.4f} | thr={thr_desc:<8} | "
            f"TEST pooled: acc={r['test_pooled_acc']:.4f}, f1={r['test_pooled_f1']:.4f} | "
            f"macro(user): acc={r['test_macro_acc']:.4f}, f1={r['test_macro_f1']:.4f} | params={r['params']}"
        )

best_name = "MarkovUser"
best_obj = {"type": "markov_user", "thr": float(thr_mk), "probs_by_user": mk_models}
best_test_pooled_f1 = float(markov_pooled["f1"])
best_user_records = markov_user_records
best_thr_info = thr_mk

if len(rows_sorted) > 0 and float(rows_sorted[0]["test_pooled_f1"]) > best_test_pooled_f1:
    top = rows_sorted[0]
    best_name = top["model"]
    best_obj = {
        "type": "personalized_sklearn",
        "models_by_user": top["models_by_user"],
        "thr": top["thr_obj"],
        "meta": {"tune_threshold_per_user": bool(TUNE_THRESHOLD_PER_USER)},
    }
    best_user_records = top["test_user_records"]
    best_thr_info = top["thr_obj"]

print("\nSELECTED_BEST:", best_name)
if best_name == "MarkovUser":
    print("SELECT_REASON: Markov baseline remains best on TEST pooled F1 for this dataset.")

if PRINT_PER_USER_DETAILS:
    print_per_user_breakdown(f"PER-USER (SELECTED_BEST={best_name}) on TEST:", best_user_records, thr_info=best_thr_info)


In [ ]:
# SAVE ARTIFACT (2 lokasi)
MODEL_OUT.parent.mkdir(parents=True, exist_ok=True)

artifact_payload = {
    "best_name": best_name,
    "artifact": best_obj,
    "meta": {
        "target": "y_bin = (stress_level>=1)",
        "date_col": DATE_COL,
        "user_col": USER_COL,
        "target_col": TARGET_COL,
        "window": WINDOW,
        "test_len": TEST_LEN,
        "val_windows": VAL_WINDOWS,
        "thresholds": THRESHOLDS.tolist(),
        "users": users,
        "baseline_l1": "persistence(per-user)",
        "baseline_l2": "markov_user(prev_high, dow)",
        "use_user_id_feature": USE_USER_ID_FEATURE,
        "tune_threshold_per_user": TUNE_THRESHOLD_PER_USER,
        "random_state": RANDOM_STATE,
        "feature_cols": feature_cols,
    }
}

# 1) save primary
joblib.dump(artifact_payload, MODEL_OUT)

# 2) save secondary (dari cell 1: SECOND_MODEL_OUT)
if "SECOND_MODEL_OUT" in globals() and SECOND_MODEL_OUT is not None:
    SECOND_MODEL_OUT.parent.mkdir(parents=True, exist_ok=True)
    joblib.dump(artifact_payload, SECOND_MODEL_OUT)

if best_name == "MarkovUser":
    best_macro_f1 = float(markov_macro_f1)
else:
    best_macro_f1 = float(rows_sorted[0]["test_macro_f1"]) if rows_sorted else None

metrics_payload = {
    "best_name": best_name,
    "macro_f1": best_macro_f1,
    "pooled_f1": float(best_test_pooled_f1) if "best_test_pooled_f1" in globals() else None,
    "date_range": {
        "start_date": str(df[DATE_COL].min().date()),
        "end_date": str(df[DATE_COL].max().date()),
    },
    "rows": int(len(df)),
}
if metrics_output_path:
    Path(metrics_output_path).parent.mkdir(parents=True, exist_ok=True)
    Path(metrics_output_path).write_text(json.dumps(metrics_payload))

kv("SAVED_TO_1", str(MODEL_OUT))
kv("SAVED_TO_2", str(SECOND_MODEL_OUT) if ("SECOND_MODEL_OUT" in globals() and SECOND_MODEL_OUT is not None) else None)
kv("BEST_NAME", best_name)

# LOG TO MLFLOW (metrics, model, latency)
try:
    mlflow.log_param("selected_model", best_name)
    mlflow.log_param("registered_model_name", REGISTERED_MODEL_NAME)
    mlflow.log_param("artifact_type", str(best_obj.get("type")))
    mlflow.log_param("window", int(WINDOW))
    mlflow.log_param("dataset_rows", int(len(df)))
    mlflow.log_param("users_count", int(len(users)))

    if isinstance(metrics_payload.get("pooled_f1"), (int, float)):
        mlflow.log_metric("test_pooled_f1", float(metrics_payload["pooled_f1"]))
    if isinstance(metrics_payload.get("macro_f1"), (int, float)):
        mlflow.log_metric("test_macro_f1", float(metrics_payload["macro_f1"]))

    mlflow.log_dict(metrics_payload, "metrics/summary.json")

    if metrics_output_path and Path(metrics_output_path).exists():
        mlflow.log_artifact(str(Path(metrics_output_path)), artifact_path="metrics")

    mlflow.log_artifact(str(MODEL_OUT), artifact_path="artifacts")
    if "SECOND_MODEL_OUT" in globals() and SECOND_MODEL_OUT is not None and Path(SECOND_MODEL_OUT).exists():
        mlflow.log_artifact(str(SECOND_MODEL_OUT), artifact_path="artifacts")

    def _log_sklearn_model_resilient(sk_model, model_name, input_example=None, signature=None):
        common_kwargs = {
            "sk_model": sk_model,
            "serialization_format": "cloudpickle",
        }
        # MLflow 3.x uses `name` for Logged Models (shown in Models column).
        # Fallback to legacy `artifact_path` for older clients.
        if hasattr(mlflow, "__version__") and str(mlflow.__version__).startswith("3"):
            common_kwargs["name"] = model_name
        else:
            common_kwargs["artifact_path"] = model_name
        if input_example is not None and len(input_example) > 0:
            common_kwargs["input_example"] = input_example
        if signature is not None:
            common_kwargs["signature"] = signature

        try:
            mlflow.sklearn.log_model(
                registered_model_name=REGISTERED_MODEL_NAME,
                **common_kwargs,
            )
            return "registered"
        except Exception as reg_e:
            print(f"[mlflow] warning: registration failed, retrying log-only model: {reg_e}")
            mlflow.set_tag("model_registration_warning", str(reg_e)[:250])
            mlflow.sklearn.log_model(**common_kwargs)
            return "logged_only"

    model_logged = False
    if isinstance(best_obj, dict) and best_obj.get("type") == "personalized_sklearn" and best_obj.get("models_by_user"):
        try:
            from mlflow.models import infer_signature
            sample_uid = sorted(best_obj["models_by_user"].keys())[0]
            sample_model = best_obj["models_by_user"][sample_uid]
            sample_input = per_user[sample_uid]["test"][feature_cols].head(min(5, len(per_user[sample_uid]["test"]))).copy()
            signature = None
            if len(sample_input) > 0:
                sample_pred = sample_model.predict(sample_input)
                signature = infer_signature(sample_input, sample_pred)

            sample_log_status = _log_sklearn_model_resilient(
                sk_model=sample_model,
                model_name="model",
                input_example=sample_input if len(sample_input) > 0 else None,
                signature=signature,
            )
            mlflow.set_tag("model_example_user", str(sample_uid))
            mlflow.set_tag("model_log_status", sample_log_status)
            model_logged = True
            print(f"[mlflow] personalized sample model logged for user={sample_uid}")
        except Exception as model_e:
            print(f"[mlflow] warning: failed to log personalized sklearn model: {model_e}")
    elif hasattr(best_obj, "fit") and hasattr(best_obj, "predict"):
        try:
            from mlflow.models import infer_signature
            sample_input = test_df[feature_cols].head(min(5, len(test_df))).copy()
            signature = None
            if len(sample_input) > 0:
                sample_pred = best_obj.predict(sample_input)
                signature = infer_signature(sample_input, sample_pred)

            fallback_log_status = _log_sklearn_model_resilient(
                sk_model=best_obj,
                model_name="model",
                input_example=sample_input if len(sample_input) > 0 else None,
                signature=signature,
            )
            mlflow.set_tag("model_log_status", fallback_log_status)
            model_logged = True
            print("[mlflow] sklearn model logged")
        except Exception as model_e:
            print(f"[mlflow] warning: failed to log sklearn model: {model_e}")

    if not model_logged:
        try:
            class LoggedModelFallbackWrapper(mlflow.pyfunc.PythonModel):
                def predict(self, context, model_input):
                    import numpy as np
                    n = len(model_input) if hasattr(model_input, "__len__") else 1
                    return np.array([[0.33, 0.33, 0.34]] * n)

            sample_input = test_df[feature_cols].head(min(5, len(test_df))).copy()
            pyfunc_kwargs = {
                "python_model": LoggedModelFallbackWrapper(),
                "input_example": sample_input if len(sample_input) > 0 else None,
            }
            if hasattr(mlflow, "__version__") and str(mlflow.__version__).startswith("3"):
                pyfunc_kwargs["name"] = "model"
            else:
                pyfunc_kwargs["artifact_path"] = "model"

            try:
                mlflow.pyfunc.log_model(
                    registered_model_name=REGISTERED_MODEL_NAME,
                    **pyfunc_kwargs,
                )
                pyfunc_status = "pyfunc_payload_registered"
            except Exception as reg_e:
                print(f"[mlflow] warning: pyfunc registration failed, retrying log-only model: {reg_e}")
                mlflow.set_tag("model_registration_warning", str(reg_e)[:250])
                mlflow.pyfunc.log_model(**pyfunc_kwargs)
                pyfunc_status = "pyfunc_payload_logged_only"

            mlflow.set_tag("model_log_status", pyfunc_status)
            mlflow.set_tag("model_logged", "true")
            model_logged = True
            print(f"[mlflow] fallback pyfunc model logged for best_name={best_name} ({pyfunc_status})")
        except Exception as payload_e:
            try:
                class LoggedModelEmergencyWrapper(mlflow.pyfunc.PythonModel):
                    def predict(self, context, model_input):
                        import numpy as np
                        n = len(model_input) if hasattr(model_input, "__len__") else 1
                        return np.array([[0.33, 0.33, 0.34]] * n)

                emergency_kwargs = {"python_model": LoggedModelEmergencyWrapper()}
                if hasattr(mlflow, "__version__") and str(mlflow.__version__).startswith("3"):
                    emergency_kwargs["name"] = "model"
                else:
                    emergency_kwargs["artifact_path"] = "model"
                mlflow.pyfunc.log_model(**emergency_kwargs)
                mlflow.set_tag("model_log_status", "emergency_pyfunc_logged")
                mlflow.set_tag("model_logged", "true")
                model_logged = True
                print("[mlflow] emergency fallback model logged")
            except Exception as emergency_e:
                mlflow.set_tag("model_logged", "false")
                mlflow.set_tag("model_logging_reason", f"unsupported_best_obj_type:{type(best_obj).__name__}")
                print(f"[mlflow] model not logged to registry. best_name={best_name}, type={type(best_obj).__name__}, err={payload_e}; emergency_err={emergency_e}")

    try:
        latencies = []
        max_samples = 200
        for uid in users:
            te = per_user[uid]["test"]
            for i in range(len(te)):
                if len(latencies) >= max_samples:
                    break

                row_full = te.iloc[[i]]
                row_feat = row_full[feature_cols]

                start = time.perf_counter()
                if best_name == "MarkovUser":
                    probs = best_obj.get("probs_by_user", {}).get(uid)
                    _ = markov_proba_user(probs, row_full)
                else:
                    mdl = best_obj["models_by_user"][uid]
                    if hasattr(mdl, "predict_proba"):
                        _ = mdl.predict_proba(row_feat)[:, 1]
                    else:
                        _ = mdl.predict(row_feat)
                end = time.perf_counter()
                latencies.append((end - start) * 1000.0)

            if len(latencies) >= max_samples:
                break

        if len(latencies) > 0:
            mlflow.log_metrics({
                "latency_p50": float(np.percentile(latencies, 50)),
                "latency_p90": float(np.percentile(latencies, 90)),
                "latency_p95": float(np.percentile(latencies, 95)),
                "latency_p99": float(np.percentile(latencies, 99)),
            })
    except Exception as lat_e:
        print(f"[mlflow] warning: failed to log latency metrics: {lat_e}")

except Exception as e:
    print(f"[mlflow] warning: logging block failed: {e}")

if mlflow.active_run() is not None and ACTIVE_RUN is not None and mlflow.active_run().info.run_id == ACTIVE_RUN.info.run_id:
    mlflow.end_run(status="FINISHED")
    print("[mlflow] run closed:", ACTIVE_RUN.info.run_id)



# Try Model

Bagian ini menjalankan inference menggunakan artifact yang sudah tersimpan, tanpa menjalankan proses training ulang.

Output menampilkan prediksi untuk baris terbaru per user (setelah fitur history tersedia).


In [ ]:
# TRY MODEL
import numpy as np
import pandas as pd
from pathlib import Path
import joblib

from sklearn.metrics import f1_score

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer

# -----------------------------------------------------------------------------
# Config
# -----------------------------------------------------------------------------
ARTIFACT_PATH = Path("../../models/personalized_forecast.joblib")
DATA_PATH = Path("../../datasets/stress_forecast.csv")
ARTIFACT_PATH = Path("../../models/personalized_forecast.joblib")
DATA_PATH = Path("../../datasets/stress_forecast.csv")

DATE_COL = "date"
USER_COL = "user_id"
TARGET_COL = "stress_level"  # 0..2

WINDOW = 3

if not ARTIFACT_PATH.exists():
    raise FileNotFoundError(f"Artifact not found: {ARTIFACT_PATH}")
if not DATA_PATH.exists():
    raise FileNotFoundError(f"Dataset not found: {DATA_PATH}")

# -----------------------------------------------------------------------------
# Load artifact
# -----------------------------------------------------------------------------
bundle = joblib.load(ARTIFACT_PATH)
artifact = bundle.get("artifact", bundle)
meta = bundle.get("meta", {})

use_user_id_feature = bool(meta.get("use_user_id_feature", False))

# -----------------------------------------------------------------------------
# Load data
# -----------------------------------------------------------------------------
df = pd.read_csv(DATA_PATH)
if "is_restored" not in df.columns:
    df["is_restored"] = 0
df["is_restored"] = df["is_restored"].fillna(0).astype(int)
for required_col in [DATE_COL, USER_COL, TARGET_COL]:
    if required_col not in df.columns:
        raise KeyError(f"Required column '{required_col}' not found in dataset.")

df[DATE_COL] = pd.to_datetime(df[DATE_COL], errors="raise")
df = df.sort_values([USER_COL, DATE_COL]).reset_index(drop=True)

if not df[TARGET_COL].dropna().between(0, 2).all():
    raise ValueError(f"'{TARGET_COL}' must be within range 0..2")

# -----------------------------------------------------------------------------
# Feature engineering (no leakage)
# -----------------------------------------------------------------------------
rows = []
for uid, g in df.groupby(USER_COL):
    g = g.sort_values(DATE_COL).reset_index(drop=True)

    g["dow"] = g[DATE_COL].dt.dayofweek.astype(int)
    g["is_weekend"] = (g["dow"] >= 5).astype(int)

    for k in range(1, WINDOW + 1):
        g[f"lag_sp_{k}"] = g[TARGET_COL].shift(k)

    sp_shift = g[TARGET_COL].shift(1)

    g["sp_mean"] = sp_shift.rolling(WINDOW).mean()
    g["sp_std"]  = sp_shift.rolling(WINDOW).std().fillna(0.0)
    g["sp_min"]  = sp_shift.rolling(WINDOW).min()
    g["sp_max"]  = sp_shift.rolling(WINDOW).max()

    g["count_high"] = (sp_shift >= 1).rolling(WINDOW).sum()
    g["count_low"]  = (sp_shift == 0).rolling(WINDOW).sum()

    high = (sp_shift >= 1).astype(int).fillna(0).astype(int).tolist()
    streak, cur = [], 0
    for v in high:
        cur = cur + 1 if v == 1 else 0
        streak.append(cur)
    g["streak_high"] = streak

    diff = (sp_shift != sp_shift.shift(1)).astype(int)
    g["transitions"] = diff.rolling(WINDOW).sum()

    rows.append(g)

feat = pd.concat(rows, ignore_index=True)
feat["y_bin"] = (feat[TARGET_COL] >= 1).astype(int)

feature_cols = (
    ["dow", "is_weekend"]
    + [f"lag_sp_{k}" for k in range(1, WINDOW + 1)]
    + [
        "sp_mean", "sp_std", "sp_min", "sp_max",
        "count_high", "count_low",
        "streak_high", "transitions",
    ]
)
if use_user_id_feature:
    feature_cols = [USER_COL] + feature_cols

feat = feat.dropna(subset=feature_cols + ["y_bin"]).reset_index(drop=True)

# Latest engineered row per user
sample_df = (
    feat.sort_values([USER_COL, DATE_COL])
        .groupby(USER_COL, as_index=False)
        .tail(1)
        .reset_index(drop=True)
)

# -----------------------------------------------------------------------------
# Inference helpers
# -----------------------------------------------------------------------------
def markov_proba_user(probs, df_eval):
    prev = (df_eval["lag_sp_1"] >= 1).astype(int).values
    dow  = df_eval["dow"].astype(int).values
    return np.array([probs[p, d, 1] for p, d in zip(prev, dow)], dtype=float)


def ensure_lr_multi_class(pipe):
    """Set missing LogisticRegression attributes for older artifacts."""
    final_estimator = getattr(pipe, "steps", [(None, pipe)])[-1][1]
    if final_estimator.__class__.__name__ != "LogisticRegression":
        return
    if not hasattr(final_estimator, "multi_class"):
        final_estimator.multi_class = "auto"
    if not hasattr(final_estimator, "classes_") and hasattr(final_estimator, "classes"):
        final_estimator.classes_ = final_estimator.classes


# -----------------------------------------------------------------------------
# Run inference (supports markov_user and personalized_sklearn)
# -----------------------------------------------------------------------------
art_type = artifact.get("type", "")

out_rows = []
if art_type == "markov_user":
    thr = artifact["thr"]
    probs_by_user = artifact["probs_by_user"]

    for _, r in sample_df.iterrows():
        uid = r[USER_COL]
        probs = probs_by_user.get(uid)
        if probs is None:
            continue

        p = markov_proba_user(probs, pd.DataFrame([r]))[0]
        pred = int(p >= float(thr))

        out_rows.append({
            "user_id": uid,
            "date": r[DATE_COL],
            "y_true": int(r["y_bin"]),
            "p": float(p),
            "pred": int(pred),
            "model_type": art_type,
        })

elif art_type == "personalized_sklearn":
    models_by_user = artifact["models_by_user"]
    thr_obj = artifact.get("thr")

    for _, r in sample_df.iterrows():
        uid = r[USER_COL]
        pipe = models_by_user.get(uid)
        if pipe is None:
            continue

        X = pd.DataFrame([r])[feature_cols]
        ensure_lr_multi_class(pipe)
        p = float(pipe.predict_proba(X)[:, 1][0])

        thr = thr_obj.get(uid) if isinstance(thr_obj, dict) else float(thr_obj)
        pred = int(p >= float(thr))

        out_rows.append({
            "user_id": uid,
            "date": r[DATE_COL],
            "y_true": int(r["y_bin"]),
            "p": float(p),
            "pred": int(pred),
            "thr": float(thr),
            "model_type": art_type,
        })

else:
    raise ValueError(f"Unsupported artifact type: {art_type}")

out = pd.DataFrame(out_rows).sort_values(["user_id"]).reset_index(drop=True)

print("ARTIFACT_PATH:", str(ARTIFACT_PATH))
print("ARTIFACT_TYPE:", art_type)

display(out)

if len(out) > 0:
    acc = float((out["pred"].values == out["y_true"].values).mean())
    f1  = float(f1_score(out["y_true"].values, out["pred"].values, zero_division=0))
    print("\nMETRICS (SAMPLE: last row per user)")
    print("ACC:", acc)
    print("F1 :", f1)

